In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

In [34]:
# 7 timesteps, 3 timesteps in de toekomst voorspellen
def create_dataset(data, input_steps=7, output_steps=3, target_col=-1):
    X, y = [], []

    # ipv data[i+input_steps:i+input_steps+output_steps, target_col]

    # y = data[target_column]
    # data = delete target_column
    # -> y[i+input_steps:i+input_steps+output_steps]

    for i in range(len(data) - input_steps - output_steps + 1):
        X.append(data[i:i+input_steps])  # alle 5 features als input
        
        y.append(
            data[i+input_steps:i+input_steps+output_steps, target_col]
        )  # slechts 1 feature als output

    return np.array(X), np.array(y)[..., np.newaxis]

In [46]:
np.random.seed(42)

timesteps = 100

t = np.arange(timesteps)

cat = np.random.choice(["A", "B", "C"], size=timesteps)


df = pd.DataFrame({
    "sin": np.sin(0.1 * t),
    "cos": np.cos(0.1 * t),
    "trend": 0.01 * t,
    "noise": np.random.randn(timesteps) * 0.1,
    "periodic": np.sin(0.05 * t) + 0.5,
    # "category": cat
})

# combined = np.column_stack((data, matches))

# # OneHotEncoder en standardscaler
# categorical_cols = ["category"]
# numerical_cols = [col for col in df.columns if col not in categorical_cols]
# preprocessor = ColumnTransformer(
#     transformers=[
#         # ("num", StandardScaler(), numerical_cols),
#         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
#     ]
# )
# data = preprocessor.fit_transform(df)
# data


X, y = create_dataset(df.values)
X[-1]
# y[-1]
# df.head(10) # -0.27176063
# df

array([[ 0.41211849, -0.91113026,  0.9       ,  0.06205482, -0.47753012],
       [ 0.31909836, -0.9477216 ,  0.91      , -0.01609374, -0.48684386],
       [ 0.22288991, -0.97484362,  0.92      , -0.03882644, -0.493691  ],
       [ 0.12445442, -0.99222533,  0.93      , -0.08855124, -0.49805444],
       [ 0.02477543, -0.99969304,  0.94      , -0.0356745 , -0.49992326],
       [-0.07515112, -0.99717216,  0.95      ,  0.05561218, -0.49929279],
       [-0.17432678, -0.98468786,  0.96      ,  0.10438606, -0.49616461]])

y is daadwerkelijk de laatste rijen. dan neem ik aan dat hij de laatste 3 rijen niet neemt in X? -> yup

In [18]:
y[0]

array([[0.84289781],
       [0.88941834],
       [0.93496553]])

In [17]:
X[0]

array([[ 0.        ,  1.        ,  0.        ,  0.05821228,  0.5       ],
       [ 0.09983342,  0.99500417,  0.01      ,  0.08877485,  0.54997917],
       [ 0.19866933,  0.98006658,  0.02      ,  0.08943323,  0.59983342],
       [ 0.29552021,  0.95533649,  0.03      ,  0.07549978,  0.64943813],
       [ 0.38941834,  0.92106099,  0.04      , -0.02071659,  0.69866933],
       [ 0.47942554,  0.87758256,  0.05      , -0.06234774,  0.74740396],
       [ 0.56464247,  0.82533561,  0.06      , -0.15081533,  0.79552021]])

# HOTENCODING

In [34]:
data = {
    "kleur": ["rood", "blauw", "groen"],
    "maat": ["S", "M", "L",],
    "prijs": [10, 15, 20]
}

df1 = pd.DataFrame(data)

# encoder aanmaken
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# alleen categorische kolommen encoden
encoded = encoder.fit_transform(df1[["kleur", "maat"]])

# kolomnamen ophalen
encoded_columns = encoder.get_feature_names_out(["kleur", "maat"])

# nieuwe dataframe maken
df_encoded = pd.DataFrame(encoded, columns=encoded_columns)

# combineren met numerieke kolommen
df_final = pd.concat([df_encoded, df1[["prijs"]]], axis=1)

df_final

,kleur_blauw,kleur_groen,kleur_rood,maat_L,maat_M,maat_S,prijs
0,0.0,0.0,1.0,0.0,0.0,1.0,10
1,1.0,0.0,0.0,0.0,1.0,0.0,15
2,0.0,1.0,0.0,1.0,0.0,0.0,20


In [ ]:
np.random.seed(42)

timesteps = 100

t = np.arange(timesteps)

cat = np.random.choice(["A", "B", "C"], size=timesteps)


df = pd.DataFrame({
    "sin": np.sin(0.1 * t),
    "cos": np.cos(0.1 * t),
    "trend": 0.01 * t,
    "noise": np.random.randn(timesteps) * 0.1,
    "periodic": np.sin(0.05 * t) + 0.5,
    "category": cat,
})

# OneHotEncoder en standardscaler
categorical_cols = ["category"]
numerical_cols = [col for col in df.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# numerieke_kollomen = df.drop(columns=categorical_cols).values

data = preprocessor.fit_transform(df)

# combined = np.column_stack((numerieke_kollomen, data))
data[0]


array([-0.28080114,  1.44633877, -1.71481604,  0.51549512, -0.20920981,
        0.        ,  0.        ,  1.        ])

In [ ]:
def make_sequences(data, window_size):
    X = []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
    return np.array(X)

window_size = 10

X_train = make_sequences(X_train_raw, window_size)
X_val   = make_sequences(X_val_raw, window_size)
X_test  = make_sequences(X_test_raw, window_size)

In [ ]:
def make_targets(data, window_size, target_col_index):
    y = []
    for i in range(len(data) - window_size):
        y.append(data[i+window_size, target_col_index])
    return np.array(y)

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

In [ ]:
# 7 timesteps, 3 timesteps in de toekomst voorspellen
def create_dataset(data, input_steps=7, output_steps=3, target_col=-1):
    X, y = [], []

    for i in range(len(data) - input_steps - output_steps + 1):
        X.append(data[i:i+input_steps])  # alle 5 features als input
        
        y.append(
            data[i+input_steps:i+input_steps+output_steps, target_col]
        )  # slechts 1 feature als output

    return np.array(X), np.array(y)[..., np.newaxis]

In [ ]:
def make_sequences(data, window_size=7): # window_size = hoeveel tijdstappen wil je mee voorspellen
    X = []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
    return np.array(X)

In [ ]:
# def make_targets(data, window_size=7, target_col_index=-1): # window_size = hoe ver in de toekomst wil je voorspellen
#     y = []
#     for i in range(len(data) - window_size):
#         y.append(data[i+window_size, target_col_index])
#     return np.array(y)

In [92]:
def make_targets(data, window_size=7, output_steps=3): # window_size = hoe ver in de toekomst wil je voorspellen
    y = []
    for i in range(len(data) - window_size - output_steps + 1):
        y.append(data[i+window_size:i+window_size+output_steps])
    return np.array(y)

In [101]:
# 1 laad de dataset in
np.random.seed(42)
timesteps = 100
t = np.arange(timesteps)

df = pd.DataFrame({
    "sin": np.sin(0.1 * t),
    "cos": np.cos(0.1 * t),
    "trend": 0.01 * t,
    "noise": np.random.randn(timesteps) * 0.1,
    "periodic": np.sin(0.05 * t) + 0.5,
    "category": np.random.choice(["A", "B", "C"], size=timesteps),
})

# 2 maak de standardscaler en hotencoding pipeline
categorical_cols = ["category"]
numerical_cols = [col for col in df.columns if col not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# 3 maak een kopie van de target kolom
target_kolom = df["periodic"].copy()

# 4 verdeel je dataset in train_df, val_df en test_df
temp_df = df[:85]
train_df = temp_df[:70]
val_df = temp_df[70:]
test_df = df[85:]

# 5 hotencode en standardscale de dfs
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

# 6 verdeel je gekopieerde kolom in y_train, y_val en y_test
y_temp_df = target_kolom[:85]
y_train = y_temp_df[:70]
y_val = y_temp_df[70:]
y_test = target_kolom[85:]

# 7 gekopieerde kolom naar numpy
y_train = y_train.values
y_val = y_val.values
y_test = y_test.values

# 8 X_train, X_val en X_test door een make_seq() functie laten gaan die enkel bedoelt is voor X
X_train = make_sequences(X_train)
X_val   = make_sequences(X_val)
X_test  = make_sequences(X_test)

# 9Q X_train, X_val en X_test door een make_seq() functie laten gaan die enkel bedoelt is voor X
y_train = make_targets(y_train)
y_val = make_targets(y_val)
y_test = make_targets(y_test)
y_train[-2] # de 7de, 8ste en 9de

array([0.34225431, 0.29309803, 0.2444589 ])

63 want 70 - 7 (-> 7 timesteps)